# Capítulo 3 — Construcción del Panel SIMCE (2015–2025)

**Tesis:** *Caída y recuperación del rendimiento SIMCE post-pandemia en establecimientos chilenos (2015–2025): trayectorias, composición socioeconómica y territorio.*

Este notebook construye, de forma **reproducible**, el panel longitudinal que alimenta el resto de la tesis. Toda la lógica vive en el paquete `simce_panel/` (funciones documentadas y reutilizables); aquí solo se **orquesta** y se **valida**.

### Qué hace el pipeline
1. **Unifica** los 4 niveles de agregación (comuna, DEPROV, región, RBD) apilando todos los años en un único archivo por nivel.
2. **Etiqueta el año** de cada fila en una columna `año`.
3. **Enriquece el panel RBD** con coordenadas (`latitud`, `longitud`) del Directorio Oficial, mediante *LEFT JOIN* por `(año, rbd)`.
4. **Genera el glosario** de los archivos unificados a partir de las glosas oficiales.

### Fuentes de datos
- Directorio de Establecimientos — MINEDUC (`datosabiertos.mineduc.cl`).
- Resultados SIMCE 2° medio — Agencia de Calidad de la Educación (`informacionestadistica.agenciaeducacion.cl`).


## 0. Configuración del entorno

Añadimos la carpeta `src/` al `path` para poder importar el paquete `simce_panel` sin instalarlo. Ajusta **`BASE_DIR`** a la carpeta que contiene tu carpeta `Dataset` (en local, Google Drive, etc.).

In [1]:
import sys
from pathlib import Path

# Limpia un import previo fallido (evita reusar la caché rota sin reiniciar el kernel).
for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]

# Busca la carpeta 'src' que contiene el paquete 'simce_panel'.
# Tu layout (carpeta 'tesis'):  Dataset/ , simce_panel/ , 01_construccion_panel.ipynb
candidatos = [
    Path.cwd() / "simce_panel" / "src",   # notebook junto a la carpeta del proyecto  <-- tu caso
    Path.cwd().parent / "src",            # notebook dentro de notebooks/
    Path.cwd() / "src",                   # notebook en la raíz del proyecto
]
RUTA_SRC = next(
    (p for p in candidatos if (p / "simce_panel" / "__init__.py").exists()),
    None,
)
if RUTA_SRC is None:
    raise FileNotFoundError(
        "No encontré 'src/simce_panel'. Ajusta RUTA_SRC a mano. Busqué en:\n  "
        + "\n  ".join(str(p) for p in candidatos)
    )
sys.path.insert(0, str(RUTA_SRC))

import simce_panel as sp
print("Paquete cargado desde:", sp.__file__)
print("simce_panel versión:", sp.__version__)

Paquete cargado desde: <ruta local>
simce_panel versión: 1.1.0


In [2]:
# >>> EDITA AQUÍ si mueves el notebook <<<
# Carpeta que contiene la carpeta 'Dataset'.
# En tu layout el notebook está JUNTO a 'Dataset', así que BASE_DIR = "."
#   (Config.desde_base le añade 'Dataset' automáticamente).
# Otros ejemplos:
#   Local:        BASE_DIR = r"C:\Users\Humberto\tesis"
#   Google Drive: BASE_DIR = "/content/drive/MyDrive/tesis"
BASE_DIR = "."

sp.configurar_logging()  # mensajes informativos en el notebook
cfg = sp.Config.desde_base(BASE_DIR)
cfg.validar()            # falla temprano si las carpetas no existen

print("Dataset:   ", cfg.dataset_dir)
print("Directorio:", cfg.directorio_dir)
print("SIMCE:     ", cfg.simce_dir)
print("Salida:    ", cfg.salida_dir)

Dataset:    Dataset
Directorio: Dataset\Directorio 2014-2025
SIMCE:      Dataset\Simce 2°medio
Salida:     Dataset\Unificados


## 1. Unificación por nivel de agregación

Para cada nivel se recorren dinámicamente las carpetas de año, se localiza el CSV correspondiente (tolerando variaciones de nombre como `_publica_fina`, `_público_final`, `_preliminar`), se añade la columna `año` y se apilan todos los años.

> Los nombres de columna equivalentes entre años (p. ej. `RBD` vs `rbd`) se **homologan automáticamente** antes de concatenar, evitando columnas duplicadas.

In [3]:
# Mapa de archivos detectados (transparencia / depuración).
mapa = sp.unificacion.mapear_archivos(cfg)
for nivel, archivos in mapa.items():
    print(f"\n{nivel.upper()}  ({len(archivos)} años):")
    for anio, ruta in archivos:
        print(f"  {anio}: {ruta.name}")


COMUNA  (8 años):
  2015: simce2m2015_comuna_publica_final.csv
  2016: simce2m2016_comuna_publica_fina.csv
  2017: simce2m2017_comuna_publica_final.csv
  2018: simce2m2018_comuna_publica_final.csv
  2022: simce2m2022_comuna_final.csv
  2023: simce2m2023_comuna_público_final.csv
  2024: simce2m2024_comuna_final.csv
  2025: simce2m2025_comuna_final.csv

DEPROV  (8 años):
  2015: simce2m2015_deprov_publica_final.csv
  2016: simce2m2016_deprov_publica_final.csv
  2017: simce2m2017_deprov_publica_final.csv
  2018: simce2m2018_deprov_publica_final.csv
  2022: simce2m2022_deprov_final.csv
  2023: simce2m2023_deprov_público_final.csv
  2024: simce2m2024_deprov_final.csv
  2025: simce2m2025_deprov_final.csv

REGION  (8 años):
  2015: simce2m2015_region_publica_final.csv
  2016: simce2m2016_region_publica_final.csv
  2017: simce2m2017_region_publica_final.csv
  2018: simce2m2018_region_publica_final.csv
  2022: simce2m2022_region_final.csv
  2023: simce2m2023_region_público_final.csv
  2024: si

In [4]:
# Ejecuta la unificación de los 4 niveles y guarda los CSV en 'Unificados'.
paneles = sp.unificar_todos(cfg, guardar=True)

15:13:55 | INFO    | simce_panel.unificacion | Unificando nivel 'comuna' (8 años)...


15:13:55 | INFO    | simce_panel.unificacion |   2015 -> 346 filas, 23 columnas


15:13:55 | INFO    | simce_panel.unificacion |   2016 -> 346 filas, 22 columnas


15:13:55 | INFO    | simce_panel.unificacion |   2017 -> 346 filas, 22 columnas


15:13:55 | INFO    | simce_panel.unificacion |   2018 -> 346 filas, 22 columnas


...

15:13:56 | INFO    | simce_panel.unificacion |   2024 -> 3000 filas, 43 columnas


15:13:56 | WARNING | simce_panel.io_datos | simce2m2025_rbd_final.csv no se pudo leer como 'utf-8-sig'; se leyó como 'cp1252'.


15:13:56 | INFO    | simce_panel.unificacion |   2025 -> 3002 filas, 43 columnas


15:13:56 | INFO    | simce_panel.unificacion |   => Panel 'rbd': 23605 filas, 61 columnas


15:13:56 | INFO    | simce_panel.unificacion |   Guardado: Dataset\Unificados\simce_rbd_unificado_2015_2025.csv


In [5]:
# Reporte consolidado: filas, columnas, años detectados y chequeo de 'año'.
sp.reporte_unificacion(paneles)

,nivel,filas,columnas,n_anios,anios,anio_ok
0,comuna,2768,29,8,"2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025",True
1,deprov,352,30,8,"2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025",True
2,region,125,28,8,"2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025",True
3,rbd,23605,61,8,"2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025",True


### 1.1 Validación — filas por año y vista previa

Confirmamos que cada año aportó filas y que la columna `año` quedó correctamente poblada (sin nulos y dentro de rango).

In [6]:
for nivel, df in paneles.items():
    print(f"\n===== {nivel.upper()} =====")
    print(sp.validacion.resumen_por_anio(df).to_string(index=False))
    chk = sp.validacion.verificar_columna_anio(df)
    print(f"Columna 'año' OK: {chk['ok']}  (nulos={chk['nulos']})")


===== COMUNA =====
 año  filas
2015    346
2016    346
2017    346
2018    346
2022    346
2023    346
2024    346
2025    346
Columna 'año' OK: True  (nulos=0)

===== DEPROV =====
 año  filas
2015     44
2016     44
2017     44
2018     44
2022     44
2023     44
2024     44
2025     44
Columna 'año' OK: True  (nulos=0)

===== REGION =====
 año  filas
2015     15
2016     15
2017     15
2018     16
2022     16
2023     16
2024     16
2025     16
Columna 'año' OK: True  (nulos=0)

===== RBD =====
 año  filas
2015   2876
2016   2902
2017   2923
2018   2935
2022   2976
2023   2991
2024   3000
2025   3002
Columna 'año' OK: True  (nulos=0)


In [7]:
# Primeras 3 filas de cada panel (columnas recortadas para legibilidad).
for nivel, df in paneles.items():
    print(f"\n===== {nivel.upper()} (primeras 3 filas) =====")
    display(sp.validacion.preview(df, n=3))


===== COMUNA (primeras 3 filas) =====


,año,agno,fecha_bbdd,codigo_bbdd,grado,cod_reg,nom_reg,cod_pro,nom_pro,cod_com,nom_com,cod_deprov
0,2015,2015,20160908,final20160908v1,2m,11.0,AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO,2.0,AISÉN,1.0,AISÉN,1.0
1,2015,2015,20160908,final20160908v1,2m,5.0,VALPARAÍSO,6.0,SAN ANTONIO,13.0,ALGARROBO,2.0
2,2015,2015,20160908,final20160908v1,2m,13.0,METROPOLITANA DE SANTIAGO,5.0,MELIPILLA,43.0,ALHUÉ,7.0



===== DEPROV (primeras 3 filas) =====


,año,agno,codigo_bbdd,fecha_bbdd,grado,cod_reg,nom_reg,cod_deprov,nom_deprov,nalu_lect2m_deprov,nalu_mate2m_deprov,nalu_soc2m_deprov
0,2015,2015,final20160908v1,20160908,2m,15,REGIÓN ARICA - PARINACOTA,151,ARICA,2466.0,2471.0,2459.0
1,2015,2015,final20160908v1,20160908,2m,15,REGIÓN ARICA - PARINACOTA,152,PARINACOTA,8.0,8.0,6.0
2,2015,2015,final20160908v1,20160908,2m,1,REGIÓN DE TARAPACÁ,11,IQUIQUE,3404.0,3488.0,3486.0



===== REGION (primeras 3 filas) =====


,año,agno,codigo_bbdd,fecha_bbdd,grado,cod_reg,nom_reg,nalu_lect2m_reg,nalu_mate2m_reg,nalu_soc2m_reg,prom_lect2m_reg,prom_mate2m_reg
0,2015,2015,final20170111v1,20170111,2m,1,REGIÓN DE TARAPACÁ,3647.0,3736.0,3734.0,242,254.0
1,2015,2015,final20170111v1,20170111,2m,2,REGIÓN DE ANTOFAGASTA,7071.0,7208.0,7162.0,243,254.0
2,2015,2015,final20170111v1,20170111,2m,3,REGIÓN DE ATACAMA,3312.0,3365.0,3359.0,236,241.0



===== RBD (primeras 3 filas) =====


[Salida omitida en la versión pública: mostraba nombres de establecimientos.]


## 2. Cruce con el Directorio — Georreferenciación del panel RBD

Se anexan `latitud` y `longitud` al panel **RBD** mediante *LEFT JOIN* por `(año, rbd)`: para los datos de cada año se usa el Directorio de ese mismo año. Se conservan **todas** las filas y columnas del SIMCE; si un RBD no tiene coordenadas en su Directorio, quedan como `NaN`.

Además de las coordenadas, pedimos algunas variables administrativas que la tesis usará como explicativas (`dependencia`, `ruralidad`). Puedes ampliar la lista con `"region"` o `"comuna"`.

In [8]:
rbd_geo = sp.enriquecer_rbd(
    paneles["rbd"],
    cfg,
    columnas_extra=["dependencia", "ruralidad"],  # opcionales; se omiten si no existen
    guardar=True,   # sobrescribe el RBD unificado con la versión enriquecida
)
paneles["rbd"] = rbd_geo

print("Columnas finales del panel RBD:")
print(list(rbd_geo.columns))

15:13:56 | INFO    | simce_panel.directorio | Directorio 2015: 15914 RBD con geo.


15:13:56 | INFO    | simce_panel.directorio | Directorio 2016: 15990 RBD con geo.


15:13:56 | INFO    | simce_panel.directorio | Directorio 2017: 16024 RBD con geo.


15:13:57 | WARNING | simce_panel.directorio | Se repararon 1 coordenadas mal formateadas.


15:13:57 | WARNING | simce_panel.directorio | 5 filas tenían latitud y longitud invertidas; se corrigieron.


...

15:13:57 | WARNING | simce_panel.directorio | Directorio 2025: 1 coordenadas requirieron atención (0 reparadas, 1 invertidas, 0 irreparables).


15:13:57 | INFO    | simce_panel.directorio | Directorio 2025: 16768 RBD con geo.


15:13:57 | INFO    | simce_panel.directorio | Cobertura de coordenadas: 99.6% de las filas RBD.


15:13:58 | INFO    | simce_panel.directorio | Guardado RBD enriquecido: Dataset\Unificados\simce_rbd_unificado_2015_2025.csv


[Salida omitida en la versión pública: mostraba nombres de establecimientos.]


In [9]:
# Cobertura de georreferenciación por año (clave para los mapas del Cap. 5).
import pandas as pd
cob = (rbd_geo
       .assign(con_geo=rbd_geo["latitud"].notna())
       .groupby("año")["con_geo"]
       .agg(["sum", "count"])
       .assign(pct=lambda d: (100 * d["sum"] / d["count"]).round(1)))
cob.columns = ["con_coords", "total", "%_cobertura"]
cob

,con_coords,total,%_cobertura
año,,,
2015,2857,2876,99.3
2016,2876,2902,99.1
2017,2891,2923,98.9
2018,2921,2935,99.5
2022,2976,2976,100.0
2023,2989,2991,99.9
2024,2999,3000,100.0
2025,3002,3002,100.0


In [10]:
# Vista previa del panel RBD georreferenciado.
sp.validacion.preview(rbd_geo[["año", "rbd", "latitud", "longitud"]], n=5, max_cols=4)

,año,rbd,latitud,longitud
0,2015,1,-18.487274,-70.294739
1,2015,4,-18.474243,-70.308258
2,2015,5,-18.476904,-70.313461
3,2015,7,-18.476147,-70.288368
4,2015,8,-18.482491,-70.317581


## 3. Glosario de los archivos unificados

Se genera un glosario único (Excel multi-hoja) que, para cada panel final, lista cada variable con su descripción (tomada de las glosas oficiales de la Agencia), su tipo de dato y su porcentaje de completitud. Las columnas derivadas (`año`, `latitud`, `longitud`) llevan descripción propia.

In [11]:
ruta_glosario = sp.exportar_glosario_excel(paneles, cfg)
print("Glosario generado en:", ruta_glosario)

15:13:58 | INFO    | simce_panel.glosario | Glosas encontradas: 8 archivo(s).


15:13:59 | INFO    | simce_panel.glosario | Glosario unificado exportado: Dataset\Unificados\glosario_unificado_2015_2025.xlsx


Glosario generado en: Dataset\Unificados\glosario_unificado_2015_2025.xlsx


In [12]:
# Vista previa del glosario del panel RBD.
glosas = sp.glosario.cargar_glosas(cfg)
sp.glosario.generar_glosario(paneles["rbd"], glosas)

15:13:59 | INFO    | simce_panel.glosario | Glosas encontradas: 8 archivo(s).


,variable,descripcion,tipo_dato,no_nulos,pct_completitud,fuente_descripcion
0,año,Año de aplicación del SIMCE (añadido por el pi...,int64,23605,100.00,derivada (pipeline)
1,agno,Año del la evaluación,int64,23605,100.00,glosas Agencia
2,fecha_bbdd,Fecha en la que se construyó esta bbdd,int64,23605,100.00,glosas Agencia
3,codigo_bbdd,Código único base de datos,str,23605,100.00,glosas Agencia
4,grado,Segundo medio,str,23605,100.00,glosas Agencia
...,...,...,...,...,...,...
60,marcadif_soc2m_rbd,Razón por la que no se muestra el puntaje en d...,str,203,0.86,glosas Agencia
61,latitud,"Latitud del establecimiento (grados decimales,...",float64,23511,99.60,derivada (pipeline)
62,longitud,Longitud del establecimiento (grados decimales...,float64,23511,99.60,derivada (pipeline)
63,dependencia,,int64,23605,100.00,sin descripción


## 4. Cómo cargar y usar el panel RBD enriquecido

Ejemplo de lectura del archivo final y cálculo del **target de la tesis**: la *diferencia de puntaje* entre el promedio post-pandemia (2022–2024) y el pre-pandemia (2015–2018) por establecimiento. Esta tabla, ya con `latitud`/`longitud`, es la base para los mapas y el modelo XGBoost + SHAP de los capítulos siguientes.

In [13]:
import pandas as pd

ruta_rbd = cfg.ruta_salida(sp.config.NOMBRES_SALIDA["rbd"])
rbd = sp.io_datos.leer_csv(ruta_rbd)   # sep=';', encoding='utf-8-sig'
print("Dimensiones:", rbd.shape)

# Localiza la columna de Matemática por FRAGMENTOS del nombre.
# (No sirve buscar por nombre exacto: la columna real se llama
#  'prom_mate2m_rbd', que no coincide con ningún alias completo.)
candidatos = sp.diagnostico.columnas_por_fragmentos(rbd, "prom", "mate", "2m")
if not candidatos:
    raise KeyError(
        "No encontré la columna de Matemática. Columnas con 'mate': "
        f"{sp.diagnostico.columnas_por_fragmentos(rbd, 'mate')}"
    )
col_mate = candidatos[0]
print("Columna de Matemática detectada:", col_mate)

Dimensiones: (23605, 65)
Columna de Matemática detectada: prom_mate2m_rbd


In [14]:
# Target ILUSTRATIVO = promedio POST (2022-2024) - promedio PRE (2015-2018).
# Nota: este cálculo es una demostración rápida, sin saneamiento de puntajes
# ni mínimo de años por periodo. El target definitivo de la tesis lo produce
# el notebook 02 (target_brecha_mate_2m.csv).
PRE  = [2015, 2016, 2017, 2018]
POST = [2022, 2023, 2024]

rbd[col_mate] = sp.io_datos.a_numerico(rbd[col_mate])

pre  = (rbd[rbd["año"].isin(PRE)]
        .groupby("rbd")[col_mate].mean().rename("mate_pre"))
post = (rbd[rbd["año"].isin(POST)]
        .groupby("rbd")[col_mate].mean().rename("mate_post"))

coords = (rbd.dropna(subset=["latitud", "longitud"])
          .sort_values("año")
          .groupby("rbd")[["latitud", "longitud"]].last())

target = (pd.concat([pre, post], axis=1)
          .assign(brecha_recuperacion=lambda d: d["mate_post"] - d["mate_pre"])
          .join(coords)
          .reset_index())

print("Establecimientos con target calculable:",
      int(target["brecha_recuperacion"].notna().sum()))
target.head()

Establecimientos con target calculable: 2866


,rbd,mate_pre,mate_post,brecha_recuperacion,latitud,longitud
0,1,197.50,242.000000,44.500000,-18.487200,-70.295212
1,4,281.25,285.000000,3.750000,-18.474239,-70.308273
2,5,285.25,261.666667,-23.583333,-18.476900,-70.313454
3,7,211.25,228.333333,17.083333,-18.476151,-70.288368
4,8,222.75,246.000000,23.250000,-18.482491,-70.317581


---
### Resumen de salidas

En la carpeta **`Dataset/Unificados/`** quedan:

| Archivo | Contenido |
|---|---|
| `simce_comuna_unificado_2015_2025.csv` | Panel por comuna, todos los años |
| `simce_deprov_unificado_2015_2025.csv` | Panel por DEPROV, todos los años |
| `simce_region_unificado_2015_2025.csv` | Panel por región, todos los años |
| `simce_rbd_unificado_2015_2025.csv` | Panel por RBD **+ coordenadas** |
| `glosario_unificado_2015_2025.xlsx` | Glosario de los 4 paneles (una hoja c/u) |

Todas las funciones usadas están documentadas en `simce_panel/` y pueden reutilizarse en los notebooks de análisis espacial y modelamiento.